# H&M Training and Validation: Six Explicit Retrieval Stages

Run one shared recommendation implementation through named, recoverable stages.
This Notebook contains configuration and orchestration; model, feature, sampling
and metric implementations live in `src/hm_recsys/`.

The sequence is **data preparation → six native retrieval channels → union/RRF
→ shared features → one LambdaMART → Validation → fresh-process reload**.
Each model cell runs only its named stage. Missing prerequisites produce an
error naming the earlier cells to run; they never trigger hidden model training.
A verified completed stage is reused only when its identity and files match.

Defaults are a bounded **smoke** run and **no final Test**. Full training requires
explicit permission, a new run ID and resource checks. A structural refactor is
not evidence of a new model-quality or speed improvement. Outputs shown after
execution are actual observations; this distributed Notebook has no saved results.

Data stays under `/content/e_commerce_recsys/data/`. Training runs locally on
Colab; verified stage artifacts persist beneath the mounted Drive data root.
Text/image vectors, their independent row maps, manifests and original indexes
are immutable inputs. They are not the learned Two-Tower index.


## 1. Run controls and identity

Use a new `RUN_ID` for this source layout. Set the Git URL and an optional full
40-character commit SHA before running. New runs may resolve `GIT_REF` once;
record the resolved SHA. `resume` and `status` require the original explicit SHA.

Only `smoke` and `full` are available. Smoke has recorded data/compute limits but
uses the same production algorithms. Full uses all legal data without silently
inheriting smoke caps. Keep `DEVICE="cpu"` for smoke. For an authorized Colab
full run, set `RUN_PROFILE="full"`, `DEVICE="cuda"` if actually available, and
`ALLOW_FULL_RUN=True`. CUDA requests fail when CUDA is unavailable.

`RUN_ACTION` applies to each individual stage: `run`, `resume`, or `status`.
Status inspects verified records; it does not run missing stages. Final Test
also needs two separate explicit flags and the locked run signature printed
only after Validation/reload. Leave all Test controls unchanged for Run All.


In [ ]:
import re
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/HaokunLiu-Vincent/ecommerce-recsys.git"
GIT_REF = "main"
GIT_COMMIT = None
GITHUB_TOKEN_SECRET = None
PROJECT_ROOT = Path("/content/e_commerce_recsys")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys")
SOURCE_DATA_ROOT = DRIVE_PROJECT_ROOT / "data"
LOCAL_DATA_ROOT = PROJECT_ROOT / "data"
RUN_PROFILE = "smoke"
RUN_ACTION = "run"
RUN_ID = "colab_smoke_modular_20261103_01"
DEVICE = "cpu"
ALLOW_FULL_RUN = False
FULL_PERFORMANCE_BACKEND = "optimized"
RUN_FINAL_TEST = False
ALLOW_FINAL_TEST = False
LOCKED_RUN_SIGNATURE = None
RECOVER_LOCK = False
SHOW_STAGE_TABLE = True
PROGRESS_POLL_SECONDS = 0.1
HEARTBEAT_SECONDS = 15

if not isinstance(RUN_ID, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID):
    raise ValueError("Use a new simple RUN_ID, not a path")
if RUN_PROFILE not in ("smoke", "full") or RUN_ACTION not in (
    "run",
    "resume",
    "status",
):
    raise ValueError("Choose smoke/full and run/resume/status")
if RUN_PROFILE == "full" and RUN_ACTION != "status" and not ALLOW_FULL_RUN:
    raise ValueError("Full requires explicit ALLOW_FULL_RUN = True")
if DEVICE not in ("cpu", "cuda") or (RUN_PROFILE == "smoke" and DEVICE != "cpu"):
    raise ValueError("Smoke requires CPU; Full device must be explicitly cpu or cuda")
if FULL_PERFORMANCE_BACKEND not in ("reference", "optimized"):
    raise ValueError("Unknown execution backend")
if LOCKED_RUN_SIGNATURE is not None and not re.fullmatch(
    r"[0-9a-f]{64}", LOCKED_RUN_SIGNATURE
):
    raise ValueError("A locked run signature must be the recorded 64-character SHA256")
if not 0.05 <= PROGRESS_POLL_SECONDS <= 2 or HEARTBEAT_SECONDS < 5:
    raise ValueError("Use bounded polling and a heartbeat of at least five seconds")

## 2. Verify a pinned GitHub checkout

The standard-library bootstrap below validates URL, ref, SHA and destination
before Git or optional secret access. It never executes downloaded remote
Python, follows a moving branch on resume, or overwrites a dirty checkout.
An existing checkout must be clean, detached and at the exact requested commit.
Use a fresh runtime for a different source revision.

For a private repository, optionally set `GITHUB_TOKEN_SECRET` to the **name** of
a Colab secret. Its value is used by a temporary askpass process, never placed
in notebook text, Git URLs, argv, configuration or displayed logs. Leave it
`None` for a public checkout. This bootstrap is longer than the stage cells
because it must run before the trusted source tree exists.


In [ ]:
import json
import os
import re
import subprocess
import tempfile
from urllib.parse import urlsplit


def validate_checkout_settings(repo_url, ref, commit, destination, *, resume=False):
    """Validate untrusted settings before any Git command or secret lookup."""
    parsed = urlsplit(repo_url)
    if (
        parsed.scheme != "https"
        or parsed.netloc.lower() != "github.com"
        or parsed.query
        or parsed.fragment
        or not re.fullmatch(
            r"/[A-Za-z0-9][A-Za-z0-9_.-]*/[A-Za-z0-9][A-Za-z0-9_.-]*", parsed.path
        )
    ):
        raise ValueError("Use a credential-free HTTPS GitHub owner/repository URL")
    if (
        not isinstance(ref, str)
        or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", ref)
        or any(part in ref for part in ("..", "//", "@{"))
        or ref.endswith(("/", "."))
        or any(
            part.startswith(".") or part.endswith(".lock") for part in ref.split("/")
        )
    ):
        raise ValueError("Use a valid branch/tag ref without Git option syntax")
    if commit is not None and (
        not isinstance(commit, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", commit)
    ):
        raise ValueError("GIT_COMMIT must be a full 40-character commit SHA")
    if resume and commit is None:
        raise ValueError("Resume/status requires the original explicit GIT_COMMIT")
    destination = Path(destination)
    if (
        not destination.is_absolute()
        or ".." in destination.parts
        or destination == Path(destination.anchor)
        or any(part.is_symlink() for part in (destination, *destination.parents))
        or (destination.exists() and not destination.is_dir())
    ):
        raise ValueError("Use a non-symlink absolute project directory")
    if destination.exists() and (not (destination / ".git").exists() or commit is None):
        raise ValueError(
            "Existing checkout requires an explicit SHA; "
            "use a fresh runtime for new code"
        )
    canonical = "https://github.com" + parsed.path.removesuffix(".git") + ".git"
    return canonical, ref, commit.lower() if commit else None, destination


def bootstrap_checkout(
    repo_url, ref, commit, destination, *, resume=False, secret_name=None
):
    """Create a fresh pinned checkout or verify an identical existing detached tree."""
    repo_url, ref, commit, destination = validate_checkout_settings(
        repo_url, ref, commit, destination, resume=resume
    )
    if secret_name is not None and not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*", secret_name
    ):
        raise ValueError("Provide a Colab secret name, never a token value")
    env = {
        **{
            k: v
            for k, v in os.environ.items()
            if not k.startswith(("GIT_CONFIG_", "GIT_TRACE"))
            and k
            not in (
                "GIT_DIR",
                "GIT_WORK_TREE",
                "GIT_INDEX_FILE",
                "GIT_OBJECT_DIRECTORY",
                "GIT_ALTERNATE_OBJECT_DIRECTORIES",
                "GIT_COMMON_DIR",
                "GIT_NAMESPACE",
                "GIT_ASKPASS",
                "SSH_ASKPASS",
                "GIT_CURL_VERBOSE",
                "HM_COLAB_GITHUB_TOKEN",
            )
        },
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_NOSYSTEM": "1",
        "GIT_CONFIG_GLOBAL": os.devnull,
    }
    with tempfile.TemporaryDirectory(prefix="hm-git-auth-") as temporary:
        if secret_name is not None:
            from google.colab import userdata

            token = userdata.get(secret_name)
            if not token or "\n" in token or "\r" in token:
                raise ValueError("The selected Colab secret is unavailable or invalid")
            askpass = Path(temporary) / "askpass.sh"
            askpass.write_text(
                '#!/bin/sh\ncase "$1" in *sername*) printf "%s" "x-access-token" ;; '
                '*) printf "%s" "$HM_COLAB_GITHUB_TOKEN" ;; esac\n',
                encoding="utf-8",
            )
            askpass.chmod(0o700)
            env.update(
                GIT_ASKPASS=str(askpass),
                GIT_ASKPASS_REQUIRE="force",
                HM_COLAB_GITHUB_TOKEN=token,
            )

        def git(*args, allowed=(0,)):
            result = subprocess.run(
                [
                    "git",
                    "-c",
                    f"core.hooksPath={os.devnull}",
                    "-c",
                    "credential.helper=",
                    "-c",
                    "core.fsmonitor=false",
                    "-c",
                    "protocol.file.allow=never",
                    *map(str, args),
                ],
                env=env,
                capture_output=True,
                text=True,
            )
            if result.returncode not in allowed:
                raise RuntimeError(
                    "Git operation failed; check repository access, "
                    "ref/SHA and secret permission"
                )
            return result.stdout.strip()

        if not destination.exists():
            print("Cloning code and resolving the requested commit...")
            git(
                "clone",
                "--no-checkout",
                "--no-recurse-submodules",
                "--filter=blob:none",
                "--depth=1",
                "--",
                repo_url,
                destination,
            )
            git(
                "-C",
                destination,
                "fetch",
                "--depth=1",
                "--no-tags",
                "--",
                "origin",
                commit or ref,
            )
            resolved = git(
                "-C", destination, "rev-parse", "--verify", "FETCH_HEAD^{commit}"
            )
            if not re.fullmatch(r"[0-9a-f]{40}", resolved) or (
                commit and resolved != commit
            ):
                raise ValueError("Fetched commit does not match the requested SHA")
            git("-C", destination, "checkout", "--detach", resolved)
        else:
            resolved = commit
        if (
            Path(git("-C", destination, "rev-parse", "--show-toplevel")).resolve()
            != destination.resolve()
        ):
            raise ValueError("Destination is not the checkout root")
        origin = git("-C", destination, "config", "--get", "remote.origin.url")
        if origin.removesuffix(".git") != repo_url.removesuffix(".git"):
            raise ValueError("Existing checkout remote differs from GITHUB_REPO_URL")
        if git("-C", destination, "status", "--porcelain", "--untracked-files=all"):
            raise ValueError("Checkout is dirty; preserve it and use a fresh runtime")
        if git("-C", destination, "rev-parse", "HEAD") != resolved:
            raise ValueError("Existing checkout SHA differs; use a fresh runtime")
        if git("-C", destination, "symbolic-ref", "-q", "HEAD", allowed=(0, 1)):
            raise ValueError(
                "Existing checkout must be detached at the explicit commit"
            )
        return resolved

In [ ]:
import importlib
import sys

CHECKOUT_COMMIT = bootstrap_checkout(
    GITHUB_REPO_URL,
    GIT_REF,
    GIT_COMMIT,
    PROJECT_ROOT,
    resume=RUN_ACTION in ("resume", "status"),
    secret_name=GITHUB_TOKEN_SECRET,
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
git_checkout = importlib.import_module("hm_recsys.runtime.git_checkout")
checkout_provenance = git_checkout.inspect_checkout(
    PROJECT_ROOT, repo_url=GITHUB_REPO_URL, ref=GIT_REF, commit=CHECKOUT_COMMIT
)
print("Verified pinned checkout:", json.dumps(checkout_provenance, indent=2))

## 3. Mount Drive and keep computation local

Mounting grants this runtime access to the existing project data directory.
Only three input directories are staged later: `original data/`,
`text_features/`, and `image_features/`. Old runs are not copied as training
inputs. Computation and large intermediate tables stay under `/content`.
Verified stage deltas and compatible checkpoints are published to
`data/runs/colab/<RUN_ID>/` on Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Prepare the locked worker environment and live display

Every stage is a separate CLI/worker invocation. The shared command helper
prints its real argv and streams flushed worker events while retaining private
logs. Counts, elapsed time, throughput, loss, epochs/batches, RSS, GPU memory and
checkpoint state come from the operation doing the work. Unknown totals stay
unknown; silence produces a heartbeat, not a fabricated progress percentage.

The existing Python 3.12/uv lock defines the worker environment. This cell does
not replace Colab's kernel PyTorch installation or download a content encoder.
The runtime probe reports actual available RAM, disk and CUDA rather than
assuming a particular Colab machine.


In [ ]:
import html
import shutil
import sys
import time

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from hm_recsys.runtime.notebook import command

LOG_ROOT = LOCAL_DATA_ROOT / "runs/colab" / RUN_ID / "notebook_logs"
PERSISTENT_ROOT = SOURCE_DATA_ROOT / "runs/colab"
progress_rows = {}
progress_display = None
progress_last_draw = 0.0


def show_progress(event: dict) -> None:
    """Show the latest actual task state for each stage, with a text-only fallback."""
    global progress_display, progress_last_draw
    if not SHOW_STAGE_TABLE:
        return
    stage = str(event.get("stage", "unknown"))
    if event.get("status") == "failed" and stage in progress_rows:
        event = {
            **progress_rows[stage],
            "status": "failed",
            "percentage": None,
            "eta_seconds": None,
            "message": event.get("message"),
        }
    progress_rows[stage] = event
    if len(progress_rows) > 64:
        progress_rows.pop(next(iter(progress_rows)))
    # Retain every observation, but redraw at most once per second between milestones.
    important = event.get("status") == "failed" or event.get("task") in (
        "stage",
        "input_files",
    )
    if not important and time.monotonic() - progress_last_draw < 1:
        return
    progress_last_draw = time.monotonic()
    rows = []
    for name, value in progress_rows.items():
        completed, total = value.get("completed"), value.get("total")
        amount = (
            f"{completed if completed is not None else 'unknown'}/"
            f"{total if total is not None else 'unknown'} {value.get('unit', '')}"
        )
        percent = value.get("percentage")
        amount += (
            f" ({percent:.1f}%)"
            if isinstance(percent, (int, float))
            else " (indeterminate)"
        )
        details = {
            "elapsed_s": value.get("elapsed_seconds", "unknown"),
            "rate_per_s": value.get("rate_per_second") or "unknown",
            "eta_s_estimate": value.get("eta_seconds")
            if value.get("eta_seconds") is not None
            else "unknown",
            "last_event": value.get("timestamp"),
            **(value.get("metrics") or {}),
        }
        columns = [
            name,
            value.get("status"),
            value.get("task"),
            amount,
            json.dumps(details, ensure_ascii=True)[:650],
        ]
        rows.append(
            "<tr>"
            + "".join(f"<td>{html.escape(str(v))}</td>" for v in columns)
            + "</tr>"
        )
    table = (
        "<table><tr><th>Stage</th><th>Latest status</th><th>Task</th>"
        "<th>Actual work</th><th>Timing / metrics</th></tr>"
        + "".join(rows)
        + "</table>"
    )
    try:
        from IPython.display import HTML, display

        if progress_display is None:
            progress_display = display(HTML(table), display_id=True)
        else:
            progress_display.update(HTML(table))
    except ImportError:
        # The shared helper's live text stream remains available outside IPython.
        return


def run_command(argv: list[str], log: Path) -> None:
    """Stream one real CLI command and preserve its complete local log."""
    command(
        argv,
        PROJECT_ROOT,
        log,
        poll_seconds=PROGRESS_POLL_SECONDS,
        heartbeat_seconds=HEARTBEAT_SECONDS,
        on_event=show_progress,
        run_id=RUN_ID,
        profile=RUN_PROFILE,
    )

In [ ]:
if shutil.disk_usage(PROJECT_ROOT).free < 25 * 1024**3:
    raise RuntimeError("Insufficient disk reserve for environment preparation")
TOOLS = Path("/content/hm_colab_tools")
run_command(
    [sys.executable, "-m", "pip", "install", "--target", str(TOOLS), "uv==0.13.6"],
    LOG_ROOT / "environment.log",
)
UV = str(TOOLS / "bin/uv")
run_command([UV, "python", "install", "3.12"], LOG_ROOT / "environment.log")
run_command([UV, "sync", "--locked", "--python", "3.12"], LOG_ROOT / "environment.log")
PYTHON = str(PROJECT_ROOT / ".venv/bin/python")
CLI = [PYTHON, str(PROJECT_ROOT / "scripts/run_pipeline.py")]
resource_probe = (
    "import json,shutil,sys,psutil,torch; "
    "print(json.dumps({'python':sys.version.split()[0], 'torch':torch.__version__, "
    "'cuda_available':torch.cuda.is_available(), 'torch_cuda':torch.version.cuda, "
    "'available_ram_bytes':psutil.virtual_memory().available, "
    "'free_disk_bytes':shutil.disk_usage('.').free}, indent=2))"
)
run_command([PYTHON, "-c", resource_probe], LOG_ROOT / "runtime.log")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

## 5. Stage and validate immutable inputs

Copy the explicit input whitelist from Drive into local `data/`, then verify
sizes/hashes and interrupted-copy recovery using the shared staging code.
Canonical preparation validates ten-character article IDs, catalog joins,
separate text/image row maps, actual embedding dimensions and availability.
Integrity failures stop affected consumers; ordinary missing content uses
masks rather than deleting future positives. Existing vectors are never
regenerated or overwritten.


In [ ]:
run_command(
    CLI
    + [
        "stage-inputs",
        "--source-root",
        str(SOURCE_DATA_ROOT),
        "--data-root",
        str(LOCAL_DATA_ROOT),
    ],
    LOG_ROOT / "staging.log",
)

## 6. Review the execution plan

All commands carry the same source, input, configuration and run identity.
The plan identifies explicit stage prerequisites and resource gates. No stage
may reuse artifacts from a pilot/old-layout run merely because their directory
exists. Reports contain aggregate diagnostics; raw customer histories and
candidate keys remain in private artifacts, not notebook output.


In [ ]:
OPTIONS = [
    "--profile",
    RUN_PROFILE,
    "--run-id",
    RUN_ID,
    "--data-root",
    str(LOCAL_DATA_ROOT),
    "--persistent-root",
    str(PERSISTENT_ROOT),
    "--mount-root",
    "/content/drive",
    "--git-remote",
    GITHUB_REPO_URL,
    "--git-ref",
    GIT_REF,
    "--git-commit",
    CHECKOUT_COMMIT,
]
if RUN_PROFILE == "full":
    OPTIONS += ["--device", DEVICE, "--performance", FULL_PERFORMANCE_BACKEND]
    if ALLOW_FULL_RUN:
        OPTIONS += ["--allow-full"]
if RECOVER_LOCK:
    OPTIONS += ["--recover-lock"]
RUN_ROOT = LOCAL_DATA_ROOT / "runs/colab" / RUN_ID
REPORT_ROOT = RUN_ROOT / "reports"
STAGE_RESULTS = {}
run_command(CLI + ["plan"] + OPTIONS, LOG_ROOT / "plan.log")
print(
    {
        "run_id": RUN_ID,
        "commit": CHECKOUT_COMMIT,
        "profile": RUN_PROFILE,
        "device": DEVICE,
        "action": RUN_ACTION,
        "local_run": str(RUN_ROOT),
        "drive_run": str(PERSISTENT_ROOT / RUN_ID),
        "final_test": RUN_FINAL_TEST,
    }
)

In [ ]:
def run_stage(stage: str, *, final_test_options: list[str] | None = None) -> None:
    """Invoke exactly one named stage; the CLI verifies prerequisites and reuse."""
    allowed = (
        "prepare",
        "trend",
        "repurchase",
        "sibling",
        "itemcf",
        "wals",
        "twotower",
        "fusion",
        "features",
        "ranker",
        "validation",
        "reload",
        "final_test",
    )
    if stage not in allowed:
        raise ValueError("Unknown stage")
    if (stage == "final_test") != bool(final_test_options):
        raise ValueError("Final Test options belong only to its explicitly gated cell")
    argv = CLI + ["stage", "--stage", stage, "--stage-action", RUN_ACTION]
    run_command(argv + OPTIONS + (final_test_options or []), LOG_ROOT / f"{stage}.log")
    STAGE_RESULTS[stage] = "status inspected" if RUN_ACTION == "status" else "success"
    print(f"Aggregate stage report: {REPORT_ROOT / (stage + '.json')}")

## 7. Data preparation: Train, Validation and Test definitions

### Train data

Prepare canonical purchase-line batches, immutable split definitions, catalog
and content lookups, cutoff-safe histories, independent query rosters and full
unique labels for **F1–F4 and Validation**. Repeated raw purchase lines remain;
labels deduplicate articles within each query. Bounded raw-ID/context arrays
support GPU-ready batches without putting the full transaction table on GPU.
The preparation report shows actual counts, dimensions, missingness and the
selected smoke/full population.

| Role | Observable cutoff / target dates | Future labels, end exclusive |
|---|---|---|
| WALS base | 2020-02-01 through 2020-08-11; vocabulary from 2020-06-16 | No future-label fit |
| Two-Tower train | Targets 2020-05-19 through 2020-08-04 | History strictly before each target day |
| Two-Tower internal selection | 2020-08-05 through 2020-08-11 | Separate from outer Validation |
| F1 | 2020-08-11 | [2020-08-12, 2020-08-19) |
| F2 | 2020-08-18 | [2020-08-19, 2020-08-26) |
| F3 | 2020-08-25 | [2020-08-26, 2020-09-02) |
| F4 | 2020-09-01 | [2020-09-02, 2020-09-09) |
| Validation | 2020-09-08 | [2020-09-09, 2020-09-16) |
| Test metadata only | 2020-09-15 | [2020-09-16, 2020-09-23) |

### Validation data

The independent Validation roster retains every eligible query, including zero
candidates and zero hits. Labels are unique future catalog-matched purchases;
active-pool and model-vocabulary misses remain in the denominator.

### Test planning only

The six outer label weeks are disjoint. Observable history is `t_dat <= cutoff`.
Test preparation here is **metadata only**: no Test future labels, scoring or
model selection. Every model below uses the documented base/cutoff history;
no channel consults its future labels while generating candidates.


In [ ]:
run_stage("prepare")

## 8. R1 — 7-Day Trend Popularity: build and validate

Build cutoff-local global/age-bucket popularity from legal seven-day history,
then generate at most **60** active candidates per query for F1–F4 and
Validation. The active pool requires at least three raw purchase lines in the
trailing 28 days; it is an activity proxy, not stock data. This rule has no
supervised fit. The stage reports rows, nonempty queries, build time and macro
Recall at available K on the full roster, including failures and zero hits.

Inputs: prepared canonical history, article/customer metadata and query rosters.
Outputs: versioned Trend candidates with original score/rank/fallback metadata
and an aggregate `reports/trend.json`. Only this channel is built here.


In [ ]:
run_stage("trend")

## 9. R2 — Time-Decayed Repurchase: build and validate

Use each user's cutoff-safe raw purchase history, preserving repeat purchases
and the existing decay/count rules. Select at most **30** active articles;
previously purchased items remain eligible. Empty history yields an honest
empty channel. The stage evaluates full unique future truth after retrieval;
truth is not restricted to retrieved or active items.

Inputs: prepared histories, active pools and independent query rosters.
Outputs: Repurchase score/rank candidates and `reports/repurchase.json`, with
per-fold rows, coverage, Recall and elapsed/throughput observations.


In [ ]:
run_stage("repurchase")

## 10. R3 — Product_Code Sibling: build and validate

Expand legal recent purchases to variants sharing the actual `product_code`,
exclude the seed article itself and preserve deterministic article-ID ties.
Apply the unchanged active pool before selecting at most **30** candidates.
This is a rule build, not a neural or supervised training job.

Inputs: canonical product-code mapping and cutoff-safe user history.
Outputs: native Sibling candidates and `reports/sibling.json`, including
per-fold/Validation coverage, counts and full-ground-truth Recall.


In [ ]:
run_stage("sibling")

## 11. R4 — Time-Decayed ItemCF: build and validate

Build or verify the cutoff-specific sparse co-purchase neighborhoods from
**global legal history**, not only the bounded model-training user subset.
Keep the asymmetric similarity, edge-date decay, neighborhood truncation and
query seed rules. Select at most **50** active articles per query.

Progress shows actual edge/pair work, cutoff builds and query batches. Resource
estimates guard pair enumeration; insufficient resources stop instead of
silently reducing the full population. Outputs include the versioned ItemCF
artifacts, native candidates and `reports/itemcf.json` for all five partitions.


In [ ]:
run_stage("itemcf")

## 12. R5 — WALS: fit, freeze, fold in and validate

Fit the existing sparse confidence-weighted ALS in its allowed base window,
or load a fully verified same-identity completed model. Freeze item factors
and the observed-item map before F1. At each cutoff, solve genuine WALS user
folding-in from current representable history with those fixed factors; pooling
is not a substitute. Do not normalize factors or filter already-liked items.

The stage shows sparse matrix shape/nnz, fit iterations, resource checks,
fold-in/query progress, candidate coverage and Recall. It outputs at most
**50** active represented items per query, learned factors/index and
`reports/wals.json`. It does not train Two-Tower or LambdaMART.


In [ ]:
run_stage("wals")

## 13. R6 — Hybrid Two-Tower: fit, freeze, encode and validate

Train the existing multi-positive LogQ model on its separate target interval;
select checkpoints using daily internal Recall@50, **not outer Validation**.
Original content vectors stay frozen, while projections and item/user weights
remain trainable. Cache only model-independent raw inputs, IDs and weights.
Prepared bounded batches retain order, history masks, full same-day positives
and the existing optimizer, FP32 and sampling definitions.

After internal selection, freeze the model, batch-encode the legal item catalog
and retrieve at most **50** active items for F1–F4/Validation. Original content
FAISS files are not this learned index. Progress includes preparation, device
transfer, epoch/batch/loss, internal validation, GPU allocated/reserved memory
and local/remote checkpoint state. `reports/twotower.json` records actual
steps/epochs, timing and outer retrieval coverage. A CUDA probe alone is not
proof that training used CUDA; actual device metadata is checked.


In [ ]:
run_stage("twotower")

## 14. Native union and RRF: merge and evaluate a rule

Merge the six persisted channel lists by query/article without future labels.
Preserve every source score, one-based rank, mask and fallback. Deduplicate
articles and compute `sum(1 / (60 + source_rank))`. The native quota sum is
**270**; there is no hidden Top-500 or RRF pre-ranking truncation.

RRF has no trainable weights. This cell evaluates its Top-12 rule baseline
alongside native union Recall and signed leave-one-channel-out contributions
without reallocating quotas. All eligible queries and full truth remain in the
denominator. Inspect candidate count distributions, overlap, zero hits and
`reports/fusion.json` before constructing features.


In [ ]:
run_stage("fusion")

## 15. Shared point-in-time ranking features

Build F1–F4 and unsampled Validation rows with the same versioned **49-column**
implementation used at inference. Row identity is query/article; raw IDs,
cutoff keys and labels are not model inputs. Preserve feature order, float32
values, categorical vocabularies, NaN/source-mask rules and content masks.
Raw content vectors are not expanded into ranker columns.

Inputs are verified native unions, frozen content/model inputs and cutoff-local
statistics. Bounded columnar batches reuse frozen visual clusters, user/cutoff
statistics and content lookups. The report shows rows/groups, schema, storage,
missingness and throughput. Resource checks cover four-fold storage and full
Validation separately. Output: feature shards plus `reports/features.json`.


In [ ]:
run_stage("features")

## 16. One LambdaMART across all four training folds

Combine F1–F4 into one grouped LambdaMART fit. Train-only sampling keeps every
native positive and selects up to 20 proxy negatives per positive using the
unchanged hard/random allocation and query-local seed. Drop only unlearnable
training groups, never zero-hit Validation queries. Assert contiguous groups,
row alignment and `sum(group_sizes) == training_rows`.

Validation uses the complete native candidate set with no negative sampling.
Candidate-list NDCG@12 is the early-stopping diagnostic; native full-ground-truth
MAP@12 is the primary end-to-end metric. No LR or second learned ranker exists.
The stage independently checks LightGBM memory needs before its single fit;
memmaps do not guarantee arbitrary-scale out-of-core training. Inspect actual
rows/positives/negatives, groups, boosting rounds, best iteration, RSS/time and
`reports/ranker.json`.


In [ ]:
run_stage("ranker")

## 17. Compare complete native Validation results

Evaluate Trend, per-channel Recall, RRF and saved LambdaMART on the same
eligible outer Validation roster and full unique truth. Annotate different
retrieval K instead of presenting quota Recall as a same-K comparison. Preserve
zero-candidate and zero-hit queries. MAP@12 divides each query's precision sum
by `min(12, full_truth_size)`; missing or inactive positives remain misses.

`reports/validation.json` separates observed metrics, query denominators and
model/schema/split identities from unrun comparisons. This cell does not fit
or tune a model, open Test labels or execute Test scoring.


In [ ]:
run_stage("validation")

## 18. Verify a fresh-process reload and lock identities

Load saved WALS/Two-Tower/ranker/mappings in isolated worker processes,
reconstruct the same Validation features and Top-12 predictions, and compare
against the reference with recorded tolerances. Do not refit or read future
Test labels. A successful file load is weaker than score/feature/order parity;
the report states what was actually compared.

Inspect `reports/reload.json` and the frozen run signature tying completed
training, Validation, source, inputs, configuration, feature schema and splits.
Only after both reports are complete may you deliberately copy that exact
signature into `LOCKED_RUN_SIGNATURE` for the separate final Test cell.


In [ ]:
run_stage("reload")

## 19. Review measured stage outcomes

Only cells whose command returned successfully appear below. For `status`,
this means an inspection ran, not that missing work completed. The CLI's
verified report is the authority for completed/reused/failed state; directory
existence is insufficient. Preserve the commit SHA and aggregate reports for
recovery. Final Test remains skipped unless its last cell is explicitly armed.


In [ ]:
print(
    json.dumps(
        {
            "run_id": RUN_ID,
            "pinned_commit": CHECKOUT_COMMIT,
            "profile": RUN_PROFILE,
            "requested_action": RUN_ACTION,
            "observed_cell_results": STAGE_RESULTS,
            "local_reports": str(REPORT_ROOT),
            "persistent_run": str(PERSISTENT_ROOT / RUN_ID),
            "final_test": "not requested"
            if not RUN_FINAL_TEST
            else "awaiting gated final cell",
        },
        indent=2,
    )
)

## Reports, recovery and next steps

Aggregate reports are under local `data/runs/colab/<RUN_ID>/reports/`; verified
artifacts/checkpoints persist under Drive
`projects/e_commerce_recsys/data/runs/colab/<RUN_ID>/`. Private per-command logs
stay under the run's `notebook_logs/`. Do not publish customer-level Parquet,
JSONL, histories, checkpoints or populated notebook outputs.

To recover, use a fresh Colab runtime, the **same explicit Git commit**, run ID,
profile/device/backend and input identity. Set `RUN_ACTION="resume"`, stage the
three immutable inputs again, and run the desired cell. Its existing
prerequisites are verified/restored from Drive; missing earlier stages produce
a named error rather than implicit training. `RUN_ACTION="status"` inspects
without creating missing stage work. A new code/config identity needs a new
run ID; do not rename old pilot/full artifacts or pull a new branch tip into a
running job. Reduced remote checkpoint frequency can lose more progress since
the last completed remote publication.

A completed smoke validates this bounded execution path, not full-data memory,
CUDA throughput or model quality. Full execution, runtime recovery and final
Test are **not run** unless actual reports show they were separately executed.
Review the current README and runbook before publishing code or starting a
larger run. Stop after the authorized milestone; no automatic full continuation,
cloud service, deployment or Test-driven search follows this Notebook.


## 20. Final Test — explicitly disabled by default

**Default Run All skips this cell without calling a worker or reading Test
future labels.** Preparing the split definition earlier does not authorize its
labels. Do not arm this cell while selecting quotas, features or checkpoints.

For a separately authorized final evaluation, use the completed **full** run,
set `RUN_FINAL_TEST=True` and `ALLOW_FINAL_TEST=True`, and copy its verified
64-character frozen signature into `LOCKED_RUN_SIGNATURE`. The backend requires
completed training/Validation/reload, matching source/input/config/schema/splits
and a locked model before reading any Test future labels. It builds only
Test-cutoff past-only candidates/features, scores without refitting, preserves
full truth and zero-hit queries, and saves a separate once-only Test report.
A mismatched or absent lock fails; it never silently tunes or refits.

This is the last executable cell. Real final Test was not run when authoring
this Notebook.


In [ ]:
if not RUN_FINAL_TEST:
    print("SKIPPED: final Test is disabled; no Test labels or scoring were requested")
else:
    if RUN_PROFILE != "full" or not ALLOW_FINAL_TEST:
        raise PermissionError(
            "Final Test requires Full and separate ALLOW_FINAL_TEST approval"
        )
    if not isinstance(LOCKED_RUN_SIGNATURE, str) or not re.fullmatch(
        r"[0-9a-f]{64}", LOCKED_RUN_SIGNATURE
    ):
        raise ValueError(
            "Set the verified frozen LOCKED_RUN_SIGNATURE after Validation/reload"
        )
    run_stage(
        "final_test",
        final_test_options=[
            "--run-final-test",
            "--approve-final-test",
            "--locked-signature",
            LOCKED_RUN_SIGNATURE,
        ],
    )